# Geometric versus electronic contributions to the band gap of ABX3 halide perovskites

Computational notebook for the B.Tech project (DTU, Engineering Physics). It runs from top to bottom
(`Runtime > Run all` on Google Colab, or `jupyter nbconvert --execute` locally).

Data: halide perovskite dataset of Yang, Manganaris and Mannodi-Kanakkithodi (Digital Discovery 2023, CC-BY),
file `perovs_data_final.xlsx` from the GitHub repository `yjq829/perovskite_dataset`.

Notes for the reader:
* Every number printed here is computed from the data in this notebook. Numbers are also collected in `numbers.json`.
* Figures go to `figures/`, tables to `tables/`.
* Mixed compounds are described by composition-weighted averages of the site properties. This is a simplification.

## Stage 0: setup, data and checks

Question: is the dataset complete and parsed correctly, and do our recomputed geometric factors match the dataset?
We download the file if needed, merge the four levels of theory, parse every formula and check the known facts.

In [1]:
# Install missing packages (needed on Colab; skipped when the package is already installed).
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pymatgen": "pymatgen",
    "scikit-learn": "sklearn",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "scipy": "scipy",
}
for pip_name, module_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print("Installing", pip_name)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

In [2]:
import json
import re
import shutil
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn
import pymatgen.core
from pymatgen.core import Element

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

NOTEBOOK_START_TIME = time.time()
print("python", sys.version.split()[0])
for module in [np, pd, matplotlib, scipy, sklearn, pymatgen.core]:
    print(module.__name__, getattr(module, "__version__", "?"))

python 3.13.18
numpy 2.5.3
pandas 3.0.5
matplotlib 3.11.2
scipy 1.18.1
sklearn 1.9.1
pymatgen.core 2026.10.31


In [3]:
# ---------------- CONFIGURATION (all paths and fixed choices in one place) ----------------
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "perovs_data_final.xlsx"
FIG_DIR = PROJECT_DIR / "figures"
TABLE_DIR = PROJECT_DIR / "tables"
EXPERIMENT_PATH = PROJECT_DIR / "experimental" / "experimental_verified.csv"
EXPERIMENT_TEMPLATE_PATH = PROJECT_DIR / "experimental_template.csv"
NUMBERS_PATH = PROJECT_DIR / "numbers.json"
DATASET_REPO_URL = "https://github.com/yjq829/perovskite_dataset.git"

RANDOM_STATE = 0
N_BOOTSTRAP = 500

FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

# Ionic radii in angstrom. Only ONE set is given ("dataset": recovered exactly from the dataset's own
# tolerance and octahedral factor columns). A second set can be added here later for the robustness test
# (stage 8); the key of the active set is ACTIVE_RADII.
RADII_SETS = {
    "dataset": {
        "A": {"K": 1.51, "Rb": 1.61, "Cs": 1.74, "Ma": 2.17, "Fa": 2.53},
        "B": {"Ge": 0.87, "Ca": 1.00, "Sn": 1.15, "Pb": 1.19, "Sr": 1.26, "Ba": 1.42},
        "X": {"Cl": 1.81, "Br": 1.96, "I": 2.20},
    },
}
ACTIVE_RADII = "dataset"

# Site elements. "Ma" and "Fa" stand for methylammonium (MA) and formamidinium (FA) after formula parsing.
A_ELEMENTS = ["Cs", "K", "Rb", "Ma", "Fa"]
B_ELEMENTS = ["Ge", "Sn", "Pb", "Ca", "Sr", "Ba"]
X_ELEMENTS = ["Cl", "Br", "I"]
GROUP14_B = ["Ge", "Sn", "Pb"]
GROUP2_B = ["Ca", "Sr", "Ba"]
ORGANIC_A = ["Ma", "Fa"]

# Atomic numbers (used as descriptors Z_B and Z_X)
Z_NUMBER = {"Ge": 32, "Sn": 50, "Pb": 82, "Ca": 20, "Sr": 38, "Ba": 56, "Cl": 17, "Br": 35, "I": 53}

# Levels of theory: sheet name -> column name used in this notebook
LEVELS = ["PBE", "HSErel", "HSErel_SOC", "HSE-PBE-SOC"]
GAP_COL = {
    "PBE": "gap_PBE",
    "HSErel": "gap_HSErel",
    "HSErel_SOC": "gap_HSErel_SOC",
    "HSE-PBE-SOC": "gap_HSE_PBE_SOC",
}
# Pretty labels for plots
A_LABEL = {"Cs": "Cs", "K": "K", "Rb": "Rb", "Ma": "MA", "Fa": "FA"}

In [4]:
# ---------------- PLOT STYLE AND COLOURS ----------------
# One colour per meaning, never reused for another meaning (checked by the assert below).
# Base palettes: Okabe-Ito (B elements) and Paul Tol palettes (colour-blind safe).
COLOR_B = {"Ge": "#E69F00", "Sn": "#56B4E9", "Pb": "#009E73",
           "Ca": "#D55E00", "Sr": "#CC79A7", "Ba": "#0072B2"}
COLOR_GROUP = {"Group 14": "#332288", "mixed B": "#BBBBBB", "Group 2": "#882255"}
COLOR_ATYPE = {"inorganic A": "#000000", "organic A": "#DDCC77", "mixed organic/inorganic A": "#44AA99"}
COLOR_LEVEL = {"PBE": "#77AADD", "HSErel": "#EE8866", "HSErel_SOC": "#AAAA00", "HSE-PBE-SOC": "#994455"}
COLOR_MODEL = {"mean baseline": "#777777", "RandomForest": "#117733", "Ridge": "#AA4499", "Linear": "#4477AA"}
COLOR_CV = {"random KFold": "#EECC66", "GroupKFold": "#004488"}
MARKER_ATYPE = {"inorganic A": "o", "organic A": "^", "mixed organic/inorganic A": "s"}

all_colours = []
for palette in [COLOR_B, COLOR_GROUP, COLOR_ATYPE, COLOR_LEVEL, COLOR_MODEL, COLOR_CV]:
    all_colours = all_colours + list(palette.values())
assert len(all_colours) == len(set(all_colours)), "a colour is used for two meanings"

plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 7.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
# ---------------- SMALL HELPER FUNCTIONS (defined once, used everywhere) ----------------
NUMBERS = {}


def to_plain(value):
    """Turn numpy numbers into plain Python numbers (for JSON)."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        if np.isnan(value):
            return None
        return round(float(value), 4)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def record(name, value, stage, note=""):
    """Store a reported number with the stage that produced it (written to numbers.json)."""
    NUMBERS[name] = {"value": to_plain(value), "stage": stage, "note": note}


def write_numbers():
    with open(NUMBERS_PATH, "w") as handle:
        json.dump(NUMBERS, handle, indent=1)
    print("numbers.json now holds", len(NUMBERS), "numbers")


def save_fig(fig, name):
    """Save a figure at 300 dpi to figures/ and show it."""
    path = FIG_DIR / (name + ".png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("saved", path.name)
    plt.show()
    plt.close(fig)


def save_table(table, name, index=False):
    path = TABLE_DIR / (name + ".csv")
    table.to_csv(path, index=index)
    print("saved table", path.name, "shape", table.shape)


def panel_label(ax, text):
    ax.text(-0.12, 1.04, text, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def eta_squared(values, labels):
    """Fraction of the variance of `values` explained by the category `labels` (one-way ANOVA eta squared)."""
    values = pd.Series(np.asarray(values, dtype=float))
    labels = pd.Series(np.asarray(labels))
    total_ss = ((values - values.mean()) ** 2).sum()
    group_means = values.groupby(labels).transform("mean")
    within_ss = ((values - group_means) ** 2).sum()
    return float(1.0 - within_ss / total_ss)


def make_model(name, seed=RANDOM_STATE):
    """The three model types used in stage 5 (and later)."""
    if name == "mean baseline":
        return DummyRegressor(strategy="mean")
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1)
    if name == "Ridge":
        return Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
    if name == "Linear":
        return LinearRegression()
    raise ValueError(name)


def cv_predict(model, X, y, splitter, groups=None):
    """Out-of-fold predictions. Each row is predicted by a model that did not see it."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    predictions = np.full(len(y), np.nan)
    fold_r2 = []
    for train_index, test_index in splitter.split(X, y, groups):
        fitted = clone(model).fit(X[train_index], y[train_index])
        predictions[test_index] = fitted.predict(X[test_index])
        fold_r2.append(r2_score(y[test_index], predictions[test_index]))
    return predictions, fold_r2


def cv_scores(model, X, y, splitter, groups=None):
    """R2 (pooled out-of-fold), mean per-fold R2 and MAE for one model and one splitter."""
    predictions, fold_r2 = cv_predict(model, X, y, splitter, groups)
    return {"R2": r2_score(y, predictions), "R2_fold_mean": float(np.mean(fold_r2)),
            "MAE": mae(y, predictions), "predictions": predictions}

print("helpers ready")

helpers ready


### Stage 0a: get the data file
If `data/perovs_data_final.xlsx` is missing, the dataset repository is cloned into a temporary folder and the file is
copied. If that fails, the notebook stops: no data is ever created or guessed.

In [6]:
if DATA_PATH.exists():
    print("Data file found:", DATA_PATH)
else:
    DATA_PATH.parent.mkdir(exist_ok=True)
    temp_folder = Path(tempfile.mkdtemp())
    print("Cloning", DATASET_REPO_URL, "into", temp_folder)
    result = subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(temp_folder / "repo")],
                            capture_output=True, text=True)
    print(result.stdout, result.stderr)
    found_files = list((temp_folder / "repo").rglob("perovs_data_final.xlsx"))
    if len(found_files) == 0:
        raise RuntimeError("Could not obtain perovs_data_final.xlsx. Please upload it to data/. "
                           "No data has been created or simulated.")
    shutil.copy(found_files[0], DATA_PATH)
    print("Copied", found_files[0], "to", DATA_PATH)

Data file found: /home/user/BTP-Perovskite/data/perovs_data_final.xlsx


### Stage 0b: load the four sheets and merge them on Formula (outer merge)

In [7]:
raw_sheets = pd.read_excel(DATA_PATH, sheet_name=None)
print("Sheets:", list(raw_sheets.keys()))
for sheet_name, sheet in raw_sheets.items():
    print(sheet_name, sheet.shape)
print("\nPBE columns:", list(raw_sheets["PBE"].columns))

# Find the deviation-of-cubicity columns by searching for "Deviation"
pbe_sheet = raw_sheets["PBE"]
DEVIATION_COLS_RAW = [c for c in pbe_sheet.columns if "Deviation" in c]
print("Deviation columns:", DEVIATION_COLS_RAW)
DEVIATION_RENAME = {
    "Deviation of Cubicity b axis": "dev_b",
    "Deviation of Cubicity c axis": "dev_c",
    "Deviation of Cubicity at alpha": "dev_alpha",
    "Deviation of Cubicity at beta": "dev_beta",
    "Deviation of Cubicity at gamma": "dev_gamma",
}
assert set(DEVIATION_RENAME) == set(DEVIATION_COLS_RAW)
DEV_COLS = list(DEVIATION_RENAME.values())

Sheets: ['PBE', 'HSErel', 'HSErel_SOC', 'HSE-PBE-SOC']
PBE (488, 12)
HSErel (299, 4)
HSErel_SOC (282, 4)
HSE-PBE-SOC (244, 4)

PBE columns: ['Formula', 'Decomposition Energy (eV p.f.u.)', 'Band Gap (eV)', 'SLME @ 5um', 'Deviation of Cubicity b axis', 'Deviation of Cubicity c axis', 'Deviation of Cubicity at alpha', 'Deviation of Cubicity at beta', 'Deviation of Cubicity at gamma', 'Octaehdral Factor', 'Tolerance Factor', "Bartel's Tolerance Factor"]
Deviation columns: ['Deviation of Cubicity b axis', 'Deviation of Cubicity c axis', 'Deviation of Cubicity at alpha', 'Deviation of Cubicity at beta', 'Deviation of Cubicity at gamma']


In [8]:
# Start from the list of all formulas, then add one gap column per level of theory.
master = pd.DataFrame({"Formula": pd.concat([s["Formula"] for s in raw_sheets.values()]).unique()})
for level in LEVELS:
    sheet = raw_sheets[level]
    assert sheet["Formula"].duplicated().sum() == 0, "duplicated formula in " + level
    gaps = sheet[["Formula", "Band Gap (eV)"]].rename(columns={"Band Gap (eV)": GAP_COL[level]})
    master = master.merge(gaps, on="Formula", how="outer")

# Extra PBE columns: deviation of cubicity, dataset octahedral/tolerance factor, dataset "Bartel" column
pbe_extra = pbe_sheet[["Formula"] + DEVIATION_COLS_RAW + ["Octaehdral Factor", "Tolerance Factor",
                                                          "Bartel's Tolerance Factor"]]
pbe_extra = pbe_extra.rename(columns=DEVIATION_RENAME)
pbe_extra = pbe_extra.rename(columns={"Octaehdral Factor": "dataset_octahedral",
                                      "Tolerance Factor": "dataset_tolerance",
                                      "Bartel's Tolerance Factor": "dataset_bartel_column"})
master = master.merge(pbe_extra, on="Formula", how="left")
master = master.sort_values("Formula").reset_index(drop=True)
print(master.shape)
master.head()

(490, 13)


,Formula,gap_PBE,gap_HSErel,gap_HSErel_SOC,gap_HSE_PBE_SOC,dev_b,dev_c,dev_alpha,dev_beta,dev_gamma,dataset_octahedral,dataset_tolerance,dataset_bartel_column
0,CsBaBr_1.125Cl_1.875,4.187,NaN,NaN,NaN,6.694269e-03,1.478023e-02,5.034304e-04,0.000044,4.532193e-07,0.760884,0.775962,6.029438
1,CsBaBr_3,3.996,5.1203,5.0142,5.0142,1.500929e-07,1.500929e-07,0.000000e+00,0.000000,0.000000e+00,0.724490,0.774052,6.029438
2,CsBaCl_3,4.667,5.9307,5.9059,5.9059,4.737084e-07,4.737084e-07,0.000000e+00,0.000000,0.000000e+00,0.784530,0.777161,6.029438
3,CsBaI_0.375_Cl_2.625,4.067,NaN,NaN,NaN,3.971800e-02,1.917515e-02,2.027437e-07,0.000282,1.887523e-06,0.763954,0.776119,6.029438
4,CsBaI_0.75_Cl_2.25,3.897,NaN,NaN,NaN,3.830729e-02,4.811735e-03,8.240648e-03,0.003483,5.705933e-03,0.744430,0.775108,6.029438


### Stage 0c: parse formulas
"MA" becomes "Ma" and "FA" becomes "Fa", then the regex `([A-Z][a-z]?)(?:_([0-9.]+))?` gives element counts
(a missing count means 1). Halide fraction = halide count / 3.

In [9]:
FORMULA_PATTERN = re.compile(r"([A-Z][a-z]?)(?:_([0-9.]+))?")
KNOWN_ELEMENTS = A_ELEMENTS + B_ELEMENTS + X_ELEMENTS


def parse_formula(formula):
    """Return a dictionary element -> count for one dataset formula, e.g. 'CsPbI_3' -> {Cs:1, Pb:1, I:3}."""
    text = formula.replace("MA", "Ma").replace("FA", "Fa")
    counts = {}
    for element, number in FORMULA_PATTERN.findall(text):
        if element not in KNOWN_ELEMENTS:
            raise ValueError("unknown element " + element + " in " + formula)
        if number == "":
            amount = 1.0
        else:
            amount = float(number)
        counts[element] = counts.get(element, 0.0) + amount
    return counts


parse_errors = []
parsed_rows = []
for formula in master["Formula"]:
    try:
        parsed_rows.append(parse_formula(formula))
    except ValueError as error:
        parse_errors.append(str(error))
        parsed_rows.append({})
print("parse errors:", len(parse_errors))
assert len(parse_errors) == 0

composition = pd.DataFrame(parsed_rows).reindex(columns=KNOWN_ELEMENTS).fillna(0.0)
# Site fractions: A and B counts already sum to 1; halide fraction = count / 3
for element in A_ELEMENTS + B_ELEMENTS:
    master["frac_" + element] = composition[element].values
for element in X_ELEMENTS:
    master["frac_" + element] = composition[element].values / 3.0

A_FRAC = ["frac_" + e for e in A_ELEMENTS]
B_FRAC = ["frac_" + e for e in B_ELEMENTS]
X_FRAC = ["frac_" + e for e in X_ELEMENTS]
a_sum = master[A_FRAC].sum(axis=1)
b_sum = master[B_FRAC].sum(axis=1)
x_sum = master[X_FRAC].sum(axis=1)
print("A-site sums: min %.12f max %.12f" % (a_sum.min(), a_sum.max()))
print("B-site sums: min %.12f max %.12f" % (b_sum.min(), b_sum.max()))
print("X-site sums: min %.12f max %.12f" % (x_sum.min(), x_sum.max()))
assert np.allclose(a_sum, 1.0) and np.allclose(b_sum, 1.0) and np.allclose(x_sum, 1.0)

parse errors: 0
A-site sums: min 1.000000000000 max 1.000000000000
B-site sums: min 1.000000000000 max 1.000000000000
X-site sums: min 1.000000000000 max 1.000000000000


### Stage 0d: labels and flags

* `g2_frac` = fraction of Ca+Sr+Ba on the B site. **Groups are defined by g2_frac** (not by the largest B element).
  `strict_group` = "Group 14" if g2_frac == 0, "Group 2" if g2_frac == 1, otherwise "mixed B".
* `A_main`, `B_main`, `X_main` = element with the largest fraction on each site (only used for grouping keys and
  colouring, never to define Group 14 / Group 2). Ties are broken by the order of the element lists above.
* `pure` = formula contains exactly one "_" (one element per site).
* `equal_flag` = HSErel_SOC gap exactly equal to HSE-PBE-SOC gap.

In [10]:
master["g2_frac"] = master[["frac_Ca", "frac_Sr", "frac_Ba"]].sum(axis=1)
master["g14_flag"] = 1.0 - master["g2_frac"]
master["strict_group"] = "mixed B"
master.loc[master["g2_frac"] == 0, "strict_group"] = "Group 14"
master.loc[master["g2_frac"] == 1, "strict_group"] = "Group 2"

master["A_main"] = master[A_FRAC].idxmax(axis=1).str.replace("frac_", "")
master["B_main"] = master[B_FRAC].idxmax(axis=1).str.replace("frac_", "")
master["X_main"] = master[X_FRAC].idxmax(axis=1).str.replace("frac_", "")
master["group_key"] = master["A_main"] + "-" + master["B_main"] + "-" + master["X_main"]

master["n_A_species"] = (master[A_FRAC] > 0).sum(axis=1)
master["n_B_species"] = (master[B_FRAC] > 0).sum(axis=1)
master["n_X_species"] = (master[X_FRAC] > 0).sum(axis=1)
master["pure"] = master["Formula"].str.count("_") == 1
master["single_B"] = master["n_B_species"] == 1
master["mixing_site"] = "pure"
master.loc[master["n_A_species"] > 1, "mixing_site"] = "A-mixed"
master.loc[master["n_B_species"] > 1, "mixing_site"] = "B-mixed"
master.loc[master["n_X_species"] > 1, "mixing_site"] = "X-mixed"

master["organic_frac"] = master["frac_Ma"] + master["frac_Fa"]
master["A_type"] = "mixed organic/inorganic A"
master.loc[master["organic_frac"] == 0, "A_type"] = "inorganic A"
master.loc[master["organic_frac"] == 1, "A_type"] = "organic A"

both_soc = master[GAP_COL["HSErel_SOC"]].notna() & master[GAP_COL["HSE-PBE-SOC"]].notna()
master["equal_flag"] = both_soc & (master[GAP_COL["HSErel_SOC"]] == master[GAP_COL["HSE-PBE-SOC"]])

# Check that the "pure" definition (exactly one "_") agrees with "one species on each site"
one_species = (master["n_A_species"] == 1) & (master["n_B_species"] == 1) & (master["n_X_species"] == 1)
assert (one_species == master["pure"]).all()
print(master["mixing_site"].value_counts())
print(master["strict_group"].value_counts())

mixing_site
B-mixed    151
A-mixed    125
X-mixed    124
pure        90
Name: count, dtype: int64
strict_group
Group 2     187
Group 14    180
mixed B     123
Name: count, dtype: int64


### Stage 0e: assertions against the facts in CLAUDE.md section 3
Count mismatches are collected (not hidden) and reported; a different file version would be listed here.

In [11]:
CHECK_RESULTS = []


def check(description, observed, expected):
    passed = observed == expected
    CHECK_RESULTS.append({"check": description, "observed": observed, "expected": expected, "passed": passed})
    print(("PASS " if passed else "FAIL ") + description + ": observed " + str(observed) + ", expected " + str(expected))


check("compounds after outer merge", len(master), 490)
check("pure compounds", int(master["pure"].sum()), 90)
check("PBE gap coverage", int(master[GAP_COL["PBE"]].notna().sum()), 488)
check("HSErel gap coverage", int(master[GAP_COL["HSErel"]].notna().sum()), 299)
check("HSErel_SOC gap coverage", int(master[GAP_COL["HSErel_SOC"]].notna().sum()), 282)
check("HSE-PBE-SOC gap coverage", int(master[GAP_COL["HSE-PBE-SOC"]].notna().sum()), 244)
no_pbe = master[master[GAP_COL["PBE"]].isna()]
check("compounds without a PBE gap", len(no_pbe), 2)
check("compounds without PBE gap that are mixed", int((~no_pbe["pure"]).sum()), 2)
check("rows with both SOC levels", int(both_soc.sum()), 244)
check("rows with equal HSErel_SOC and HSE-PBE-SOC", int(master["equal_flag"].sum()), 41)
check("equal rows in strict Group 14", int((master["equal_flag"] & (master["strict_group"] == "Group 14")).sum()), 0)
present_A = sorted([e for e in A_ELEMENTS if master["frac_" + e].sum() > 0])
present_B = sorted([e for e in B_ELEMENTS if master["frac_" + e].sum() > 0])
present_X = sorted([e for e in X_ELEMENTS if master["frac_" + e].sum() > 0])
check("A-site species", present_A, sorted(A_ELEMENTS))
check("B-site species", present_B, sorted(B_ELEMENTS))
check("X-site species", present_X, sorted(X_ELEMENTS))
check("parse errors", len(parse_errors), 0)
fapb = master.set_index("Formula").loc[["FAPbI_3", "FAPbCl_3"], GAP_COL["HSE-PBE-SOC"]]
check("FAPbI3 and FAPbCl3 have no HSE-PBE-SOC gap", bool(fapb.isna().all()), True)
print("\nCompounds without a PBE gap:")
print(no_pbe[["Formula"] + list(GAP_COL.values())].to_string(index=False))
print("\nEqual-flag rows by strict group:")
print(master[master["equal_flag"]]["strict_group"].value_counts())
check_table = pd.DataFrame(CHECK_RESULTS)
save_table(check_table.astype(str), "stage0_checks")
n_failed = int((~check_table["passed"]).sum())
print("failed checks:", n_failed)
record("n_compounds", len(master), "stage 0")
record("n_pure", int(master["pure"].sum()), "stage 0")
for level in LEVELS:
    record("coverage_" + level, int(master[GAP_COL[level]].notna().sum()), "stage 0")
record("n_equal_flag", int(master["equal_flag"].sum()), "stage 0")
record("n_failed_checks", n_failed, "stage 0")

PASS compounds after outer merge: observed 490, expected 490
PASS pure compounds: observed 90, expected 90
PASS PBE gap coverage: observed 488, expected 488
PASS HSErel gap coverage: observed 299, expected 299
PASS HSErel_SOC gap coverage: observed 282, expected 282
PASS HSE-PBE-SOC gap coverage: observed 244, expected 244
PASS compounds without a PBE gap: observed 2, expected 2
PASS compounds without PBE gap that are mixed: observed 2, expected 2
PASS rows with both SOC levels: observed 244, expected 244
PASS rows with equal HSErel_SOC and HSE-PBE-SOC: observed 41, expected 41
PASS equal rows in strict Group 14: observed 0, expected 0
PASS A-site species: observed ['Cs', 'Fa', 'K', 'Ma', 'Rb'], expected ['Cs', 'Fa', 'K', 'Ma', 'Rb']
PASS B-site species: observed ['Ba', 'Ca', 'Ge', 'Pb', 'Sn', 'Sr'], expected ['Ba', 'Ca', 'Ge', 'Pb', 'Sn', 'Sr']
PASS X-site species: observed ['Br', 'Cl', 'I'], expected ['Br', 'Cl', 'I']
PASS parse errors: observed 0, expected 0
PASS FAPbI3 and FAPbCl3 

### Stage 0f: features
Composition-weighted averages for mixed compounds. Radii from the active radii set (config cell).
Electronic descriptors (Pauling electronegativity, first and second ionization energies) are read from `pymatgen`.

In [12]:
# Electronic properties from pymatgen (not typed by hand)
element_property_rows = []
for element in B_ELEMENTS + X_ELEMENTS:
    pmg = Element(element)
    ionization = pmg.ionization_energies
    element_property_rows.append({"element": element, "Z": pmg.Z, "EN_pauling": float(pmg.X),
                                  "IE1_eV": float(ionization[0]), "IE2_eV": float(ionization[1]),
                                  "IE1_plus_IE2_eV": float(ionization[0] + ionization[1])})
ELEMENT_PROPS = pd.DataFrame(element_property_rows).set_index("element")
# The Z values in the config must agree with pymatgen
for element in B_ELEMENTS + X_ELEMENTS:
    assert ELEMENT_PROPS.loc[element, "Z"] == Z_NUMBER[element]
save_table(ELEMENT_PROPS.reset_index(), "element_properties_pymatgen")
ELEMENT_PROPS

saved table element_properties_pymatgen.csv shape (9, 6)


,Z,EN_pauling,IE1_eV,IE2_eV,IE1_plus_IE2_eV
element,,,,,
Ge,32,2.01,7.899435,15.934610,23.834045
Sn,50,1.96,7.343918,14.633070,21.976988
Pb,82,2.33,7.416680,15.032499,22.449179
Ca,20,1.00,6.113155,11.871719,17.984874
Sr,38,0.95,5.694867,11.030276,16.725144
Ba,56,0.89,5.211665,10.003826,15.215491
Cl,17,3.16,12.967633,23.813640,36.781273
Br,35,2.96,11.813810,21.591000,33.404810
I,53,2.66,10.451260,19.131260,29.582520


In [13]:
def build_features(table, radii):
    """Add geometric and electronic descriptors to a copy of `table` using the radii dictionary `radii`."""
    out = table.copy()
    out["rA"] = 0.0
    out["rB"] = 0.0
    out["rX"] = 0.0
    out["Z_B"] = 0.0
    out["Z_X"] = 0.0
    out["EN_B"] = 0.0
    out["EN_X"] = 0.0
    out["IE_B"] = 0.0
    for element in A_ELEMENTS:
        out["rA"] = out["rA"] + out["frac_" + element] * radii["A"][element]
    for element in B_ELEMENTS:
        weight = out["frac_" + element]
        out["rB"] = out["rB"] + weight * radii["B"][element]
        out["Z_B"] = out["Z_B"] + weight * Z_NUMBER[element]
        out["EN_B"] = out["EN_B"] + weight * ELEMENT_PROPS.loc[element, "EN_pauling"]
        out["IE_B"] = out["IE_B"] + weight * ELEMENT_PROPS.loc[element, "IE1_plus_IE2_eV"]
    for element in X_ELEMENTS:
        weight = out["frac_" + element]
        out["rX"] = out["rX"] + weight * radii["X"][element]
        out["Z_X"] = out["Z_X"] + weight * Z_NUMBER[element]
        out["EN_X"] = out["EN_X"] + weight * ELEMENT_PROPS.loc[element, "EN_pauling"]
    out["dEN"] = out["EN_X"] - out["EN_B"]
    out["octahedral"] = out["rB"] / out["rX"]
    out["tolerance"] = (out["rA"] + out["rX"]) / (np.sqrt(2.0) * (out["rB"] + out["rX"]))
    # Bartel et al. 2019: tau = rX/rB - nA * (nA - (rA/rB) / ln(rA/rB)), nA = oxidation state of A = 1
    n_A = 1.0
    ratio = out["rA"] / out["rB"]
    out["bartel_term"] = ratio / np.log(ratio)
    out["bartel_tau"] = out["rX"] / out["rB"] - n_A * (n_A - out["bartel_term"])
    return out


features = build_features(master, RADII_SETS[ACTIVE_RADII])

GEOM_FEATURES = ["rA", "rB", "rX", "octahedral", "tolerance"]
Z_FEATURES = ["Z_B", "Z_X"]
ELEC_FEATURES = ["EN_B", "EN_X", "IE_B", "dEN"]
print("rA > rB for every compound (needed for the ln in Bartel tau):", bool((features["rA"] > features["rB"]).all()))

rA > rB for every compound (needed for the ln in Bartel tau): True


In [14]:
# Reproduce the dataset's own octahedral and tolerance factor columns (rows that have a PBE entry)
has_dataset_factors = features["dataset_tolerance"].notna()
tolerance_diff = (features.loc[has_dataset_factors, "tolerance"] - features.loc[has_dataset_factors, "dataset_tolerance"]).abs().max()
octahedral_diff = (features.loc[has_dataset_factors, "octahedral"] - features.loc[has_dataset_factors, "dataset_octahedral"]).abs().max()
bartel_term_diff = (features.loc[has_dataset_factors, "bartel_term"] - features.loc[has_dataset_factors, "dataset_bartel_column"]).abs().max()
print("max |t - dataset t|      = %.2e (n = %d)" % (tolerance_diff, has_dataset_factors.sum()))
print("max |mu - dataset mu|    = %.2e" % octahedral_diff)
print("max |(rA/rB)/ln(rA/rB) - dataset 'Bartel' column| = %.2e" % bartel_term_diff)
assert tolerance_diff < 1e-12 and octahedral_diff < 1e-12
print("\nThe dataset 'Bartel' column equals only the term (rA/rB)/ln(rA/rB); it is NOT used anywhere.")
print("Bartel tau (recomputed): min %.3f, max %.3f; compounds with tau < 4.18: %d of %d"
      % (features["bartel_tau"].min(), features["bartel_tau"].max(), (features["bartel_tau"] < 4.18).sum(), len(features)))
record("max_abs_diff_tolerance", tolerance_diff, "stage 0")
record("max_abs_diff_octahedral", octahedral_diff, "stage 0")
record("max_abs_diff_bartel_term", bartel_term_diff, "stage 0")
record("n_bartel_tau_below_4p18", int((features["bartel_tau"] < 4.18).sum()), "stage 0")

max |t - dataset t|      = 2.22e-16 (n = 488)
max |mu - dataset mu|    = 1.11e-16
max |(rA/rB)/ln(rA/rB) - dataset 'Bartel' column| = 1.78e-14

The dataset 'Bartel' column equals only the term (rA/rB)/ln(rA/rB); it is NOT used anywhere.
Bartel tau (recomputed): min 3.315, max 17.853; compounds with tau < 4.18: 230 of 490


In [15]:
# Save the master table (data + labels) and the feature table
master_columns = ["Formula"] + list(GAP_COL.values()) + DEV_COLS + ["dataset_octahedral", "dataset_tolerance",
                  "dataset_bartel_column"] + A_FRAC + B_FRAC + X_FRAC + ["g2_frac", "g14_flag", "strict_group",
                  "A_main", "B_main", "X_main", "group_key", "pure", "single_B", "mixing_site", "organic_frac",
                  "A_type", "equal_flag"]
save_table(features[master_columns], "master_table")
feature_columns = ["Formula", "strict_group", "g2_frac", "g14_flag", "organic_frac"] + GEOM_FEATURES + \
                  ["bartel_tau"] + Z_FEATURES + ELEC_FEATURES + A_FRAC + B_FRAC + X_FRAC + DEV_COLS
save_table(features[feature_columns], "features")
summary_counts = features.groupby("strict_group").agg(
    n_all=("Formula", "size"),
    n_PBE=(GAP_COL["PBE"], "count"),
    n_HSErel=(GAP_COL["HSErel"], "count"),
    n_HSErel_SOC=(GAP_COL["HSErel_SOC"], "count"),
    n_HSE_PBE_SOC=(GAP_COL["HSE-PBE-SOC"], "count"),
)
save_table(summary_counts.reset_index(), "stage0_counts_by_group")
print(summary_counts)
for group_name in summary_counts.index:
    record("n_HSE_PBE_SOC_" + group_name.replace(" ", "_"), summary_counts.loc[group_name, "n_HSE_PBE_SOC"], "stage 0")
write_numbers()

saved table master_table.csv shape (490, 40)


saved table features.csv shape (490, 36)


saved table stage0_counts_by_group.csv shape (3, 6)
              n_all  n_PBE  n_HSErel  n_HSErel_SOC  n_HSE_PBE_SOC
strict_group                                                     
Group 14        180    178        88            82             76
Group 2         187    187        94            89             77
mixed B         123    123       117           111             91
numbers.json now holds 15 numbers
